In [2]:
import csv
from collections import defaultdict
 
 
def load_csv(path):
    """Just reads the CSV into a header + list of rows, skipping blanks."""
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [row for row in reader if row]
    return header, rows
 
 
def ask_for_the_new_record(header, rows):
    """
    Walks the user through entering a value for every column
    (except the last one, since that's what we're predicting).
 
    For employee_data.csv, this is where you'd type in things like
    the Gender, Department, Designation, and City of the person
    you want to classify.
    """
    columns_to_ask_about = header[:-1]
    new_record = {}
 
    print("\nOkay, let's build the record you want to classify.")
    print("-" * 60)
    for i, column in enumerate(columns_to_ask_about):
        seen_values = sorted(set(row[i] for row in rows))
        print(f"{column} - here's what shows up in the data: {seen_values}")
        answer = input(f"  What's the {column}? ").strip()
        new_record[column] = answer
 
    return new_record
 
 
def naive_bayes(header, rows, new_record):
    columns_to_ask_about = header[:-1]
    target_column = header[-1]  # e.g. "Remote Eligible"
    total_rows = len(rows)
 
    print("\n" + "=" * 60)
    print(f"Columns we're using: {columns_to_ask_about}")
    print(f"What we're predicting: '{target_column}'")
    print("=" * 60)
 
    # First, split everyone up by their class.
    # For our employee example: who's Remote Eligible, who isn't.
 
    grouped_by_class = defaultdict(list)
    for row in rows:
        grouped_by_class[row[-1]].append(row)
 
    print("Step 1: How many rows fall into each class")
    for label, group in grouped_by_class.items():
        print(f"  {target_column} = {label}: {len(group)} rows")
    print("=" * 60)
 
    # Priors: before looking at any details, what's the base rate
    # for each class? E.g. what fraction of employees are Remote
    # Eligible = Yes, just going by the raw counts?

    priors = {label: len(group) / total_rows
              for label, group in grouped_by_class.items()}
 
    print("Step 2: Prior probabilities (the base rates)")
    for label, p in priors.items():
        n = len(grouped_by_class[label])
        print(f"  P({target_column}={label}) = {n}/{total_rows} = {p:.4f}")
    print("=" * 60)
 

    # Now, for each column, how likely is the value the user gave us,
    # within each class? If we've never seen that combination before,
    # we smooth it a bit instead of just calling it impossible.
    # --------------------------------------------------------------
    print("Step 3: How likely is each answer, within each class?")
    print(f"The record we're classifying: {new_record}")
    print("-" * 60)
 
    likelihood_pieces = {label: [] for label in grouped_by_class}
    for column in columns_to_ask_about:
        col_index = header.index(column)
        answer = new_record[column]
        for label, group in grouped_by_class.items():
            matches = sum(1 for row in group if row[col_index] == answer)
            group_size = len(group)
            if matches == 0:
                # never seen this exact value in this class before,
                # so we lean on Laplace smoothing instead of a hard zero
                unique_values = len(set(row[col_index] for row in rows))
                probability = 1 / (group_size + unique_values)
                note = "  (smoothed - never saw this value in this class)"
            else:
                probability = matches / group_size
                note = ""
            likelihood_pieces[label].append(probability)
            print(f"  P({column}={answer}|{target_column}={label}) = "
                  f"{matches}/{group_size} = {probability:.4f}{note}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # The "naive" part: we pretend all the columns are independent
    # of each other and just multiply their probabilities together.
    # --------------------------------------------------------------
    print("Step 4: Multiply those probabilities together (the naive part)")
    combined_likelihood = {}
    for label, probabilities in likelihood_pieces.items():
        product = 1.0
        for p in probabilities:
            product *= p
        combined_likelihood[label] = product
        chain = " * ".join(f"{p:.4f}" for p in probabilities)
        print(f"  P(record|{target_column}={label}) = {chain} = {product:.6f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Combine each class's likelihood with its prior to get a score.
    # --------------------------------------------------------------
    print("Step 5: Fold in the prior for each class")
    scores = {}
    for label in grouped_by_class:
        scores[label] = combined_likelihood[label] * priors[label]
        print(f"  {combined_likelihood[label]:.6f} * {priors[label]:.4f} "
              f"= {scores[label]:.6f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Turn the raw scores into proper probabilities that add up to 1.
    # --------------------------------------------------------------
    total_score = sum(scores.values())
    print(f"Step 6: Normalize so everything adds up to 1 "
          f"({' + '.join(f'{s:.6f}' for s in scores.values())} = {total_score:.6f})")
    for label, s in scores.items():
        posterior = s / total_score
        print(f"  P({target_column}={label}|record) = {s:.6f}/{total_score:.6f} "
              f"= {posterior:.4f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Whichever class scored highest wins.
    # --------------------------------------------------------------
    winner = max(scores, key=scores.get)
    print(f"Step 7: And the winner is... {target_column} = {winner}")
    return winner
 
 
if __name__ == "__main__":
    csv_path = input("Path to your CSV (e.g. employee_data.csv): ").strip()
    header, rows = load_csv(csv_path)
    print(f"\nLoaded {len(rows)} rows from '{csv_path}'.")
    new_record = ask_for_the_new_record(header, rows)
    naive_bayes(header, rows, new_record)

Path to your CSV (e.g. employee_data.csv):  employee_data.csv



Loaded 200 rows from 'employee_data.csv'.

Okay, let's build the record you want to classify.
------------------------------------------------------------
S.No - here's what shows up in the data: ['1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '18', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '19', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '2', '20', '200', '21', '22', '23', '

  What's the S.No?  


Name - here's what shows up in the data: ['Abhishek Shetty', 'Ajay Shetty', 'Ajay Yadav', 'Amit Kumar', 'Amit Pillai', 'Anil Das', 'Anil Kumar', 'Anil Pillai', 'Anil Shetty', 'Anita Kumar', 'Anita Malhotra', 'Anita Murthy', 'Anita Pandey', 'Arjun Menon', 'Arjun Naidu', 'Arjun Sharma', 'Ashok Menon', 'Ashok Trivedi', 'Bhavana Bhatt', 'Bhavana Bose', 'Bhavana Mishra', 'Bhavana Murthy', 'Bhavana Trivedi', 'Bhavana Yadav', 'Chitra Bhatt', 'Chitra Kapoor', 'Chitra Naidu', 'Chitra Pandey', 'Deepak Chowdary', 'Deepak Gupta', 'Deepak Iyer', 'Deepak Kapoor', 'Deepak Singh', 'Deepika Menon', 'Divya Bhatt', 'Divya Bose', 'Divya Chatterjee', 'Divya Choudhury', 'Divya Desai', 'Divya Gupta', 'Divya Kapoor', 'Divya Mishra', 'Divya Nair', 'Divya Sharma', 'Divya Yadav', 'Ganesh Desai', 'Ganesh Murthy', 'Geetha Kapoor', 'Geetha Mishra', 'Gopal Kapoor', 'Gopal Mishra', 'Gopal Shetty', 'Harish Bose', 'Harish Gupta', 'Harish Iyer', 'Harish Malhotra', 'Harish Shetty', 'Jyothi Chowdary', 'Jyothi Gupta', 'Jyo

  What's the Name?  Gopal Kapoor


...

Salary (INR) - here's what shows up in the data: ['100353', '102110', '102607', '103907', '103986', '104276', '104299', '104457', '104569', '105654', '106006', '106560', '107210', '108238', '108611', '108816', '108904', '109109', '110215', '110477', '111079', '111163', '111341', '111706', '111761', '112500', '112878', '113039', '113184', '113878', '113984', '114399', '114928', '115023', '115710', '115921', '116303', '117104', '119203', '119861', '120448', '121068', '121530', '121810', '121835', '122120', '122735', '122754', '123366', '123567', '124503', '124943', '125290', '126361', '127123', '128364', '128544', '128952', '129771', '129842', '130396', '130620', '130987', '131318', '132785', '133794', '136376', '136848', '137108', '137674', '137917', '137960', '139572', '139955', '140075', '140898', '140970', '143187', '143639', '144079', '144806', '146098', '146591', '147619', '148067', '148347', '149105', '25772', '29244', '29288', '29799', '30224', '30273', '30706', '31572', '31590',

  What's the Salary (INR)?  100353


City - here's what shows up in the data: ['Ahmedabad', 'Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']


  What's the City?  Delhi



Columns we're using: ['S.No', 'Name', 'Gender', 'Age', 'Department', 'Designation', 'Experience (Years)', 'Salary (INR)', 'City']
What we're predicting: 'Remote Eligible'
Step 1: How many rows fall into each class
  Remote Eligible = Yes: 97 rows
  Remote Eligible = No: 103 rows
Step 2: Prior probabilities (the base rates)
  P(Remote Eligible=Yes) = 97/200 = 0.4850
  P(Remote Eligible=No) = 103/200 = 0.5150
Step 3: How likely is each answer, within each class?
The record we're classifying: {'S.No': '', 'Name': 'Gopal Kapoor', 'Gender': 'Female', 'Age': '44', 'Department': 'Sales', 'Designation': 'Analyst', 'Experience (Years)': '10', 'Salary (INR)': '100353', 'City': 'Delhi'}
------------------------------------------------------------
  P(S.No=|Remote Eligible=Yes) = 0/97 = 0.0034  (smoothed - never saw this value in this class)
  P(S.No=|Remote Eligible=No) = 0/103 = 0.0033  (smoothed - never saw this value in this class)
  P(Name=Gopal Kapoor|Remote Eligible=Yes) = 0/97 = 0.0035  (